Read from the silver table


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window 

In [0]:
query = """ 
SELECT 
ROW_NUMBER() OVER ( ORDER BY ci.customer_id) AS customer_key,
ci.customer_id, 
ci.customer_number,
ci.first_name,
ci.last_name,
lo.country,
ci.marital_status,
CASE 
    WHEN ci.gender <> 'n/a' THEN ci.gender
    ELSE COALESCE(ca.gender, 'n/a')
END AS gender, 
ca.birth_date AS birthdate, 
ci.created_date AS create_date
FROM databricksprash.silver_layer_data.customer_info ci
LEFT JOIN databricksprash.silver_layer_data.erp_customers ca
ON ci.customer_number = ca.customer_number
LEFT JOIN databricksprash.silver_layer_data.erp_customer_location lo
ON ci.customer_number = lo.customer_number
"""
df = spark.sql(query)


Sanity Check

In [0]:
df.limit(10).display()

### Writing to Gold Table


In [0]:
(
    df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("gold_layer_data.dim_customers")
)

Sanity Checks 

In [0]:

%sql
SELECT * FROM databricksprash.gold_layer_data.dim_customers LIMIT 10;